In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn import tree

from sklearn.metrics import confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report
from sklearn.tree import plot_tree
import pydotplus
import graphviz

In [2]:
# To display multiple outputs from the same cell use the following codes
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

# Change the width of the current notebook
from IPython.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

In [3]:
# Importing dataset and putting it in a variable
filename = "DissertationCombinedDataset.xlsx"
df = pd.read_excel(filename)
print("Dataset shape: ", df.shape)
print("Sample data of dataset:")
df.head()

Dataset shape:  (1000, 257)
Sample data of dataset:


,PropertyAddress,Postcode,UPRN,FullName,SaleRole,SellDate,CompanyNumber,DirectorName,CountryOfIncorp,Section1,...,FittingsReplaced,OtherFittings,KeysAndCodes,Section14,ConsentsCopies,ConsentsAttached&ToFollow,ConsentsNotAvailable,FurtherInformationDetail,FurtherInformationDocuments,Section15
0,Address1,ZV74 9SV,260322496806,John 1,Seller(s),14/09/2025,30887982,Jane 1,United Kingdom,NaN,...,Yes,Yes,Yes,NaN,", To follow, Not applicable, Not available",& Consents to follow 1,Consents not available 1,NaN,Not applicable,NaN
1,Address2,FY35 5TI,366741746515,John 2,Trustee(s),26/09/2024,RM383591,Jane 2,United Kingdom,NaN,...,No,Yes,Yes,NaN,", To follow, ,",& Consents to follow 2,NaN,NaN,Not applicable,NaN
2,Address3,Y63 1CG,48309584706,John 3,Seller(s),09/06/2026,VB903674,Jane 3,United Kingdom,NaN,...,No,No,No,NaN,", , Not applicable,",&,NaN,NaN,Not applicable,NaN
3,Address4,P28 5GY,367940705206,John 4,Executor(s)/Administrator(s),20/01/2026,RH452247,Jane 4,United Kingdom,NaN,...,No,Yes,Yes,NaN,"Attached, , , Not available",Consents attached 4 &,Consents not available 4,NaN,Attached,NaN
4,Address5,T96 8PN,182063213078,John 5,Seller(s),26/09/2025,31772266,Jane 5,United Kingdom,NaN,...,Yes,No,No,NaN,"Attached, To follow, ,",Consents attached 5 & Consents to follow 5,NaN,NaN,Not applicable,NaN


In [4]:
# Checking the default data type
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Columns: 257 entries, PropertyAddress to Section15
dtypes: float64(28), int64(1), object(228)
memory usage: 2.0+ MB


In [5]:
df.dtypes

PropertyAddress                 object
Postcode                        object
UPRN                             int64
FullName                        object
SaleRole                        object
                                ...   
ConsentsAttached&ToFollow       object
ConsentsNotAvailable            object
FurtherInformationDetail        object
FurtherInformationDocuments     object
Section15                      float64
Length: 257, dtype: object

In [6]:
# store columns with specific data type
integer_columns = df.select_dtypes(include=['int64']).columns
float_columns = df.select_dtypes(include=['float64']).columns
object_columns = df.select_dtypes(include=['object']).columns

# display columns
print('\nint64 columns:\n',integer_columns)
print('\nfloat64 columns:\n',float_columns)
print('\nobject columns:\n',object_columns)


int64 columns:
 Index(['UPRN'], dtype='object')

float64 columns:
 Index(['Section1', 'Section2', 'Section3', 'Section4', 'Section5Alt',
       'InstallYear', 'Section5Solar', 'Section5Other', 'Section6', 'Section7',
       'Section8Flood', 'Section8Other', 'Section9Rights', 'Section9Drain',
       'Section10', 'Section11Elec', 'Section11Heat', 'Section11Sewer',
       'Section12Elec', 'MPRN', 'Section12Gas', 'Section12Water',
       'Section12Sewage', 'Section12Heat', 'Section12Tele', 'Section13',
       'Section14', 'Section15'],
      dtype='object')

object columns:
 Index(['PropertyAddress', 'Postcode', 'FullName', 'SaleRole', 'SellDate',
       'CompanyNumber', 'DirectorName', 'CountryOfIncorp', 'LeftBoundary',
       'RightBoundary',
       ...
       'SalePriceMortage', 'RubbishRemoved', 'FittingsReplaced',
       'OtherFittings', 'KeysAndCodes', 'ConsentsCopies',
       'ConsentsAttached&ToFollow', 'ConsentsNotAvailable',
       'FurtherInformationDetail', 'FurtherInformation

In [7]:
# UPRN has data type int64 and all of the output column and MPRN have data type float64

In [8]:
# Since this dataset does not involve any numerical calculation, every columns should be a string
for column in df.columns:
    if df[column].dtype == 'float64':
        df[column] = df[column].astype(str)
    elif df[column].dtype == 'int64':
        df[column] = df[column].astype(str)

In [9]:
# Double-checking the new data types of the columns now
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Columns: 257 entries, PropertyAddress to Section15
dtypes: object(257)
memory usage: 2.0+ MB


In [10]:
# Replacing all of NaN with empty string
df.fillna('', inplace=True)

# Section 1 Rules

In [11]:
for i, x in enumerate(df['Section1']):
    # If the value inside these columns are empty, the section is declared "Invalid"
    if (df.loc[i, ['PropertyAddress', "FullName", "SellDate"]] == '').any():
        df.loc[i, 'Section1'] = 'Invalid'
    else:
        df.loc[i, 'Section1'] = 'Valid'

In [12]:
df['Section1'].value_counts()

Section1
Valid      561
Invalid    439
Name: count, dtype: int64

# Section 2 Rules

In [13]:
# These are the important section 2 columns that should not be empty
impsect2 = ['LeftBoundary', 'RightBoundary', 'RearBoundary', 'FrontBoundary', 'IrregularBoundary', 'BoundaryChanged', 'OverUnderHang', 'PartyWall']

for i, x in enumerate(df['Section2']):
    # If these section 2 columns have empty value, it's invalid
    if (df.loc[i, impsect2] == '').any():
        df.loc[i, 'Section2'] = "Invalid"

    # If the answer is yes and no explanation was given, invalid
    elif (df.loc[i, "BoundaryChanged"] == 'Yes') & (df.loc[i, "ChangeDetail"] == ''): 
        df.loc[i, 'Section2'] = "Invalid"
        
    elif (df.loc[i, "OverUnderHang"] == 'Yes') & (df.loc[i, "OverUnderDetail"] == ''):
        df.loc[i, 'Section2'] = "Invalid"
        
    elif (df.loc[i, "PartyWall"] == 'Yes') & (df.loc[i, "PartyWallDetail"] == ''): 
        df.loc[i, 'Section2'] = "Invalid"
    
    # If any of these boundary are not known, it's follow-up check
    elif (df.loc[i, "LeftBoundary"] == "Not Known") | (df.loc[i, "RightBoundary"] == "Not Known") | (df.loc[i, "RearBoundary"] == "Not Known") | (df.loc[i, "FrontBoundary"] == "Not Known"):
        df.loc[i, 'Section2'] = "Follow-up Check"

    # If the answer is "To follow", follow-up check
    elif df.loc[i, "IrregularBoundary"] == "To follow":
        df.loc[i, 'Section2'] = "Follow-up Check"
    
    # If the answer is "Not known", follow-up check
    elif df.loc[i, "OverUnderHang"] == 'Not known':
        df.loc[i, 'Section2'] = "Follow-up Check"

    # If answer is "Yes" and following answer is 'To follow', follow-up check
    elif (df.loc[i, "PartyWall"] == 'Yes') & (df.loc[i, "PartyWallNotice"] == 'To follow'): 
        df.loc[i, 'Section2'] = "Follow-up Check"
        
    else:
        df.loc[i, 'Section2'] = "Valid"

In [14]:
df['Section2'].value_counts()

Section2
Invalid            478
Follow-up Check    466
Valid               56
Name: count, dtype: int64

# Section 3 Rules

In [15]:
for i, x in enumerate(df['Section3']):
    # If any of these two columns are empty, it's invalid
    if (df.loc[i, ['PropertyDispute', 'PossibleDispute']] == '').any():
        df.loc[i, 'Section3'] = "Invalid"
        
    # If the answer is yes and no explanation is given, invalid
    elif (df.loc[i, 'PropertyDispute'] == 'Yes') & (df.loc[i, 'DisputeDetail'] == ''):
        df.loc[i, 'Section3'] = "Invalid"
        
    elif (df.loc[i, 'PossibleDispute'] == 'Yes') & (df.loc[i, 'PossibleDetail'] == ''):
        df.loc[i, 'Section3'] = "Invalid"
        
    else:
        df.loc[i, 'Section3'] = "Valid"

In [16]:
df['Section3'].value_counts()

Section3
Valid      629
Invalid    371
Name: count, dtype: int64

# Section 4 Rules

In [17]:
for i, x in enumerate(df['Section4']):
    # If any of these 3 columns are empty, it's invalid
    if (df.loc[i, ['PropertyComm', 'PropertyPlan', 'PropertyAlt']] == '').any():
        df.loc[i, 'Section4'] = 'Invalid'
        
    # If the answer is yes and no explanation is given, invalid
    elif (df.loc[i, 'PropertyComm'] ==  'Yes') & (df.loc[i, 'CommDetail'] == ''):
        df.loc[i, 'Section4'] = 'Invalid'
        
    elif (df.loc[i, 'PropertyPlan'] ==  'Yes') & (df.loc[i, 'PlanDetail'] == ''):
        df.loc[i, 'Section4'] = 'Invalid'
        
    elif (df.loc[i, 'PropertyAlt'] ==  'Yes') & (df.loc[i, 'AltDetail'] == ''):
        df.loc[i, 'Section4'] = 'Invalid'
        
    else:
        df.loc[i, 'Section4'] = 'Valid'

In [18]:
df['Section4'].value_counts()

Section4
Valid      575
Invalid    425
Name: count, dtype: int64

# Section 5 Rules

## Section 5 Alt Section

In [19]:
# These are the important section 5 alt columns that should not be empty
impsect5_1 = ['WindowAlt', 'ExtensionAlt', 'ConserAlt', 'LoftAlt', 'GarageAlt', 'IntWallAlt', 'ChimneyAlt', 'InsulationAlt', 'OtherAlt', 'NonResidentPurpose', 'PermBreach', 'PlanIssue']

# These are the multiple choices questions sub-section of section 5 alt
altsect5choice = ['WindowAlt', 'ExtensionAlt', 'ConserAlt', 'LoftAlt', 'GarageAlt', 'IntWallAlt', 'ChimneyAlt', 'InsulationAlt', 'OtherAlt']

# These are the explanation section if ANY of the previous multiple choices are answered "Yes"
altsect5explain = ['AltDetail', 'AltComplete', 'PermCopy', 'PermAcquire']

In [20]:
for i, x in enumerate(df['Section5Alt']):
    # if any of these columns are empty, invalid
    if (df.loc[i, impsect5_1] == '').any():
        df.loc[i, 'Section5Alt'] = 'Invalid'
        
    # if any of these sections are answered "Yes" and the following sections are empty, invalid
    elif ((df.loc[i, altsect5choice] == 'Yes').any()) & ((df.loc[i, altsect5explain] == '').any()):
        df.loc[i, 'Section5Alt'] = 'Invalid'
        
    # if answer is "No" and no explanation given, invalid
    elif (df.loc[i, 'AltComplete'] == 'No') & (df.loc[i, 'CompleteDetail'] == ''):
        df.loc[i, 'Section5Alt'] = 'Invalid'

    # if answer is "No" and no explanation given, invalid
    elif (df.loc[i, 'PermAcquire'] == 'No') & (df.loc[i, 'PermDetail'] == ''):
        df.loc[i, 'Section5Alt'] = 'Invalid'

    # if answer is "Yes" and no explanation given, invalid
    elif (df.loc[i, 'NonResidentPurpose'] == 'Yes') & (df.loc[i, 'NonResidentDetail'] == ''):
        df.loc[i, 'Section5Alt'] = 'Invalid'

    # if answer is "Yes" and no explanation given, invalid
    elif (df.loc[i, 'PermBreach'] == 'Yes') & (df.loc[i, 'BreachDetail'] == ''):
        df.loc[i, 'Section5Alt'] = 'Invalid'

    # if answer is "Yes" and no explanation given, invalid
    elif (df.loc[i, 'PlanIssue'] == 'Yes') & (df.loc[i, 'IssueDetail'] == ''):
        df.loc[i, 'Section5Alt'] = 'Invalid'
    
    # if answer is "Not available", follow-up check
    elif df.loc[i, 'PermCopy'] == 'Not available':
        df.loc[i, 'Section5Alt'] = 'Follow-up Check'

    # if answer is 'To follow', follow-up check
    elif df.loc[i, 'PermCopy'] == 'To follow':
        df.loc[i, 'Section5Alt'] = 'Follow-up Check'
        
    # if answer is "Yes" and answer is "To follow", follow-up check
    elif (df.loc[i, 'NonResidentPurpose'] == 'Yes') & (df.loc[i, 'NonResidentCopy'] == 'To follow'):
        df.loc[i, 'Section5Alt'] = 'Follow-up Check'
        
    else:
        df.loc[i, 'Section5Alt'] = 'Valid'

In [21]:
df['Section5Alt'].value_counts()

Section5Alt
Invalid            777
Follow-up Check    174
Valid               49
Name: count, dtype: int64

## Section 5 Solar Section

In [22]:
# If there is a solar power system, these questions columns should not be empty
impsect5_2 = ['HeatOnly', 'InstallYear', 'SystemOwn', 'SolarLease', 'MaintenanceAgreement', 'SolarBattery', 'NationalGrid', 'SolarCertCopy']

# If there is a FIT or SEG in place, these columns should not be empty
fit_seg_question = ['FITCopy', 'ElecBillCopy', 'FITDetail']

In [23]:
for i, x in enumerate(df['Section5Solar']):
    # if this column is empty, invalid
    if (df.loc[i, 'SolarInstalled'] == ''):
        df.loc[i, 'Section5Solar'] = 'Invalid'

    # if solar power was installed and any of these questions are empty, invalid
    elif (df.loc[i, 'SolarInstalled'] == 'Yes') & ((df.loc[i, impsect5_2] == '').any()):
        df.loc[i, 'Section5Solar'] = 'Invalid'
        
    # if answer is "Yes" and no description is given, invalid
    elif (df.loc[i, "SolarBattery"] == 'Yes') & (df.loc[i, "BatteryDetal"] == ''):
        df.loc[i, 'Section5Solar'] = 'Invalid'

    # if answer is "Yes" and no follow-up answer is given, invalid
    elif (df.loc[i, 'NationalGrid'] == 'Yes') & (df.loc[i, "FITSEG"] == ''):
        df.loc[i, 'Section5Solar'] = 'Invalid'

    elif (df.loc[i, 'FITSEG'] == 'Yes') & ((df.loc[i, fit_seg_question] == '').any()):
        df.loc[i, 'Section5Solar'] = 'Invalid'

    # if answer is "Not available", follow-up check
    elif (df.loc[i, 'SolarCertCopy'] == 'Not available'):
        df.loc[i, 'Section5Solar'] = 'Follow-up Check'

    # if answer is "To follow", follow-up check
    elif (df.loc[i, 'LeaseCopy'] == 'To follow'):
        df.loc[i, 'Section5Solar'] = 'Follow-up Check'

    elif (df.loc[i, "AgreementCopy"] == 'To follow'):
        df.loc[i, 'Section5Solar'] = 'Follow-up Check'

    # if any of these columns have the answer "To follow", follow-up check
    elif (df.loc[i, fit_seg_question] == 'To follow').any():
        df.loc[i, 'Section5Solar'] = 'Follow-up Check'

    # if answer is "To follow", follow-up check
    elif df.loc[i, 'SolarCertCopy'] == 'To follow':
        df.loc[i, 'Section5Solar'] = 'Follow-up Check'

    else:
        df.loc[i, 'Section5Solar'] = 'Valid'

In [24]:
df['Section5Solar'].value_counts()

Section5Solar
Valid              473
Invalid            317
Follow-up Check    210
Name: count, dtype: int64

## Section 5 Other Section

In [25]:
# These columns should have an answer or not have the answer "Not known"
impsect5_3 = ['PropertyListed', 'PropertyConservation', 'PropertyTree']

In [26]:
for i, x in enumerate(df['Section5Other']):
    # if any of these columns are empty, invalid
    if ((df.loc[i, impsect5_3] == '').any()):
        df.loc[i, 'Section5Other'] = 'Invalid'

    # if answer is yes and any of the following columns are empty, invalid
    elif (df.loc[i, 'TreeWork'] == 'Yes') & ((df.loc[i, ['TreeDetail', 'TreeCopy']] == '').any()):
        df.loc[i, 'Section5Other'] = 'Invalid'
    
    # if any of these columns have the answer "Not known", follow-up check
    elif (df.loc[i, impsect5_3] == 'Not known').any():
        df.loc[i, 'Section5Other'] = 'Follow-up Check'
        
    # if answer is "Not known", follow-up check
    elif (df.loc[i, 'TreeWork'] == 'Not known'):
        df.loc[i, 'Section5Other'] = 'Follow-up Check'

    # if answer is "To follow", follow-up check
    elif (df.loc[i, 'TreeCopy'] == 'To follow'):
        df.loc[i, 'Section5Other'] = 'Follow-up Check'

    else:
        df.loc[i, 'Section5Other'] = 'Valid'

In [27]:
df['Section5Other'].value_counts()

Section5Other
Follow-up Check    544
Invalid            291
Valid              165
Name: count, dtype: int64

# Section 6 Rules

In [28]:
# important Section 6 columns that shouldn't be empty
impsect6 = ['HomeWarrant', 'DampWarrant', 'TimberWarrant', 'WindowWarrant', 'RoofingWarrant', 'HeatWarrant', 'UnderpinWarrant', 'InsulationWarrant', 'OtherWarrant', 'WarrantAware', 'WarrantBreach']

# Section 6 columns that shouldn't have "To follow" as an answer
impsect6_2 = ['HomeCopy', 'DampCopy', 'TimberCopy', 'WindowCopy', 'RoofingCopy', 'HeatCopy', 'UnderpinCopy', 'InsulationCopy', 'OtherCopy']

In [29]:
for i, x in enumerate(df['Section6']):
    # if any of these columns are empty, invalid
    if (df.loc[i, impsect6] == '').any():
        df.loc[i, 'Section6'] = 'Invalid'

    # if answer is yes and following column is empty, invalid
    elif (df.loc[i, 'OtherWarrant'] == 'Yes') & (df.loc[i, 'OtherDetail'] == ''):
        df.loc[i, 'Section6'] = 'Invalid'

    elif (df.loc[i, 'WarrantAware'] == 'Yes') & (df.loc[i, 'WarrantDetail'] == ''):
        df.loc[i, 'Section6'] = 'Invalid'

    elif (df.loc[i, 'WarrantBreach'] == 'Yes') & (df.loc[i, 'WarrantBreachDetail'] == ''):
        df.loc[i, 'Section6'] = 'Invalid'
    
    # if any of these columns have "To follow" as an answer, follow-up check
    elif (df.loc[i, impsect6_2] == 'To follow').any():
        df.loc[i, 'Section6'] = 'Follow-up Check'

    else:
        df.loc[i, 'Section6'] = 'Valid'

In [30]:
df['Section6'].value_counts()

Section6
Invalid            495
Follow-up Check    467
Valid               38
Name: count, dtype: int64

# Section 7 Rules

In [31]:
for i, x in enumerate(df['Section7']):
    # if any of these columns are empty, invalid
    if (df.loc[i, ['PropertyInsure', 'InsureDifficulty', 'InsuranceClaims']] == '').any():
        df.loc[i, 'Section7'] = 'Invalid'

    # if the answer is "Yes" (or "No" for 7.1) and the following column is empty, invalid
    elif (df.loc[i, 'PropertyInsure'] == 'No') & (df.loc[i, 'WhoInsure'] == ''):
        df.loc[i, 'Section7'] = 'Invalid'

    elif (df.loc[i, 'InsureDifficulty'] == 'Yes') & (df.loc[i, 'DifficultyDetail'] == ''):
        df.loc[i, 'Section7'] = 'Invalid'

    elif (df.loc[i, 'InsuranceClaims'] == 'Yes') & (df.loc[i, 'ClaimsDetail'] == ''):
        df.loc[i, 'Section7'] = 'Invalid'

    else:
        df.loc[i, 'Section7'] = 'Valid'

In [32]:
df['Section7'].value_counts()

Section7
Valid      564
Invalid    436
Name: count, dtype: int64

# Section 8

## Section 8 Flood

In [33]:
# The types of flood and detail columns that should not be empty if property was ever flooded
sect8floods = ['GroundWater', 'SewerFlood', 'SurfaceWater', 'CoastalFlooding', 'RiverFlooding', 'OtherFlood', 'FloodDetail']

In [34]:
for i, x in enumerate(df['Section8Flood']):
    # if any of these columns are empty, invalid
    if (df.loc[i, ['PropertyFlood', 'FloodDefense']] == '').any():
        df.loc[i, 'Section8Flood'] = 'Invalid'

    # if answer is "Yes" and any of the following columns are empty, invalid
    elif (df.loc[i, 'PropertyFlood'] == 'Yes') & ((df.loc[i, sect8floods] == '').any()):
        df.loc[i, 'Section8Flood'] = 'Invalid'

    # if answer is "Yes" and no detail given, invalid
    elif (df.loc[i, 'FloodDefense'] == "Yes") & (df.loc[i, 'DefenseDetail'] == ''):
        df.loc[i, 'Section8Flood'] = 'Invalid'
        
    # if answer is "Not known", follow-up check
    elif df.loc[i, 'FloodDefense'] == 'Not known':
        df.loc[i, 'Section8Flood'] = 'Follow-up Check'

    else:
        df.loc[i, 'Section8Flood'] = 'Valid'

In [35]:
df['Section8Flood'].value_counts()

Section8Flood
Valid              417
Invalid            393
Follow-up Check    190
Name: count, dtype: int64

## Section 8 Other

In [36]:
# important section 8 columns that shouldn't be empty
impsect8 = ['RadonTest', 'RadonMeasures', 'GreenDeal', 'PropertyKnotweed', 'KnotweedSurvey']

In [37]:
for i, x in enumerate(df['Section8Other']):
    # if any of these columns are empty, invalid
    if (df.loc[i, impsect8] == '').any():
        df.loc[i, 'Section8Other'] = 'Invalid'

    # if answer is "Yes" and following columns are empty, invalid
    elif (df.loc[i, 'RadonTest'] == 'Yes') & ((df.loc[i, ['RadonCopy', 'TestBelow']] == '').any()):
        df.loc[i, 'Section8Other'] = 'Invalid'

    elif (df.loc[i, 'GreenDeal'] == 'Yes') & ((df.loc[i, ['DealDetails', 'DealElecBillCopy']] == '').any()):
         df.loc[i, 'Section8Other'] = 'Invalid'

    elif (df.loc[i, 'PropertyKnotweed'] == 'Yes') & (df.loc[i, 'KnotweedTreatment'] == ''):
        df.loc[i, 'Section8Other'] = 'Invalid'

    elif (df.loc[i, 'KnotweedTreatment'] == 'Yes') & (df.loc[i, 'KnotweedCopy'] == ''):
        df.loc[i, 'Section8Other'] = 'Invalid'

    elif (df.loc[i, 'KnotweedSurvey'] == 'Yes') & (df.loc[i, 'SurveyCopy'] == ''):
        df.loc[i, 'Section8Other'] = 'Invalid'
    
    # if any of these columns have the answer "Not known", follow-up check
    elif (df.loc[i, ['RadonTest', 'RadonMeasures', 'PropertyKnotweed', 'KnotweedSurvey']] == 'Not known').any():
        df.loc[i, 'Section8Other'] = 'Follow-up Check'

    # if answer is "Yes" and following columns have the answer "To follow", follow-up check
    elif (df.loc[i, 'RadonTest'] == 'Yes') & (df.loc[i, 'RadonCopy'] == 'To follow'):
        df.loc[i, 'Section8Other'] = 'Follow-up Check'

    elif (df.loc[i, 'GreenDeal'] == 'Yes') & (df.loc[i, 'DealElecBillCopy'] == 'To follow'):
        df.loc[i, 'Section8Other'] = 'Follow-up Check'

    # if answer is "Yes" and following column is "Not known", follow-up check
    elif (df.loc[i, 'PropertyKnotweed'] == 'Yes') & (df.loc[i, 'KnotweedTreatment'] == 'Not known'):
        df.loc[i, 'Section8Other'] = 'Follow-up Check'

    # if answer is "Yes" and following column is "To follow", follow-up check
    elif (df.loc[i, 'KnotweedTreatment'] == 'Yes') & (df.loc[i, 'KnotweedCopy'] == 'To follow'):
        df.loc[i, 'Section8Other'] = 'Follow-up Check'

    elif (df.loc[i, 'KnotweedSurvey'] == 'Yes') & (df.loc[i, 'SurveyCopy'] == 'To follow'):
        df.loc[i, 'Section8Other'] = 'Follow-up Check'

    else:
        df.loc[i, 'Section8Other'] = 'Valid'

In [38]:
df['Section8Other'].value_counts()

Section8Other
Follow-up Check    528
Invalid            444
Valid               28
Name: count, dtype: int64

In [39]:
# Double checking to see if "Valid" exists in the output column
check4 = [i for i, x in enumerate(df['Section8Other']) if x == 'Valid']
print(check4)

[5, 16, 19, 79, 108, 125, 136, 137, 138, 157, 214, 215, 228, 234, 259, 363, 369, 381, 383, 394, 405, 438, 459, 479, 497, 576, 745, 939]


# Section 9 Rules

## Section 9 Rights

In [40]:
# important section 9 rights columns that shouldn't be empty
impsect9 = ['ClientRights', 'JointContribute', 'RightDisagreement', 'OtherOwnerRights', 'OtherJointContribute', 'OtherRightDisagreement']

In [41]:
for i, x in enumerate(df['Section9Rights']):
    # if any of these columns are empty, invalid
    if (df.loc[i, impsect9] == '').any():
        df.loc[i, 'Section9Rights'] = 'Invalid'

    # if the answer is "Yes" and no explanation given, invalid
    elif (df.loc[i, 'ClientRights'] == 'Yes') & (df.loc[i, 'RightsDetail'] == ''):
        df.loc[i, 'Section9Rights'] = 'Invalid'

    elif (df.loc[i, 'JointContribute'] == 'Yes') & (df.loc[i, 'JointDetail'] == ''):
        df.loc[i, 'Section9Rights'] = 'Invalid'

    elif (df.loc[i, 'RightDisagreement'] == 'Yes') & (df.loc[i, 'DisagreementDetail'] == ''):
        df.loc[i, 'Section9Rights'] = 'Invalid'

    elif (df.loc[i, 'OtherOwnerRights'] == 'Yes') & (df.loc[i, 'OtherOwnerDetail'] == ''):
        df.loc[i, 'Section9Rights'] = 'Invalid'

    elif (df.loc[i, 'OtherJointContribute'] == 'Yes') & (df.loc[i, 'OtherJointDetail'] == ''):
        df.loc[i, 'Section9Rights'] = 'Invalid'

    elif (df.loc[i, 'OtherRightDisagreement'] == 'Yes') & (df.loc[i, 'OtherDisagreementDetail'] == ''):
        df.loc[i, 'Section9Rights'] = 'Invalid'
    
    # if answer is "Not Known", follow-up check
    elif df.loc[i, 'RightDisagreement'] == 'Not known':
        df.loc[i, 'Section9Rights'] = 'Follow-up Check'

    elif df.loc[i, 'OtherRightDisagreement'] == 'Not known':
        df.loc[i, 'Section9Rights'] = 'Follow-up Check'

    else:
        df.loc[i, 'Section9Rights'] = 'Valid'

In [42]:
df['Section9Rights'].value_counts()

Section9Rights
Invalid            487
Follow-up Check    301
Valid              212
Name: count, dtype: int64

## Section 9 Drain

In [43]:
for i, x in enumerate(df['Section9Drain']):
    # if any of these columns' answer is empty, invalid
    if ((df.loc[i, ['DrainCross', 'OtherDrainCross', 'DrainAgreement']] == '').any()) & ((df.loc[i, ['DrainCross', 'OtherDrainCross', 'DrainAgreement']] == 'Not known').any()):
        df.loc[i, 'Section9Drain'] = 'Invalid'

    # if answer is "Yes" and the following columns are empty, invalid
    elif (df.loc[i, 'DrainAgreement'] == 'Yes') & ((df.loc[i, ['DrainDetail', 'DrainCopy']] == '').any()):
        df.loc[i, 'Section9Drain'] = 'Invalid'

    # if answer is "Not known" for an of these columns, follow-up check
    elif (df.loc[i, ['DrainCross', 'OtherDrainCross', 'DrainAgreement']] == 'Not known').any():
        df.loc[i, 'Section9Drain'] = 'Follow-up Check'
        
    # if answer is "Yes" and the following column answer's is "To follow", follow-up check
    elif (df.loc[i, 'DrainAgreement'] == 'Yes') & (df.loc[i, 'DrainCopy'] == 'To follow'):
        df.loc[i, 'Section9Drain'] = 'Follow-up Check'

    else:
        df.loc[i, 'Section9Drain'] = 'Valid'

In [44]:
df['Section9Drain'].value_counts()

Section9Drain
Follow-up Check    513
Valid              277
Invalid            210
Name: count, dtype: int64

In [45]:
# Double checking to see if "Invalid" exists in the output column
check5 = [i for i, x in enumerate(df['Section9Drain']) if x == 'Invalid']
print(check5)

[500, 501, 502, 503, 506, 507, 511, 512, 514, 522, 523, 528, 529, 530, 531, 536, 537, 539, 540, 542, 543, 544, 547, 549, 550, 551, 552, 553, 554, 555, 556, 561, 565, 569, 573, 574, 581, 583, 586, 590, 591, 592, 593, 596, 597, 598, 599, 600, 606, 607, 608, 613, 616, 624, 625, 627, 628, 629, 632, 637, 638, 640, 642, 643, 644, 654, 657, 659, 660, 664, 670, 672, 673, 676, 678, 681, 684, 687, 692, 695, 697, 700, 701, 711, 712, 714, 717, 718, 720, 723, 724, 725, 726, 729, 730, 731, 732, 733, 735, 736, 741, 744, 745, 751, 752, 754, 756, 757, 758, 760, 763, 764, 766, 767, 769, 779, 780, 782, 783, 785, 790, 791, 792, 795, 802, 806, 807, 808, 809, 811, 813, 818, 819, 820, 821, 827, 829, 831, 839, 840, 841, 842, 845, 846, 848, 850, 853, 854, 857, 858, 862, 864, 865, 866, 867, 869, 870, 871, 873, 877, 878, 882, 883, 885, 888, 889, 892, 894, 896, 898, 900, 901, 903, 904, 909, 910, 914, 915, 916, 920, 922, 926, 927, 932, 933, 939, 941, 946, 949, 951, 952, 954, 957, 960, 961, 962, 968, 970, 971, 972,

# Section 10 Rules

In [46]:
for i, x in enumerate(df['Section10']):
    # if these columns are empty, invalid
    if (df.loc[i, ['ParkingDetail', 'ParkPermit', 'EVCharger']] == '').any():
        df.loc[i, 'Section10'] = 'Invalid'

    # if answer is "Yes" and following columns are empty, invalid
    elif (df.loc[i, 'EVCharger'] == 'Yes') & ((df.loc[i, ['EVChargerDetail', 'EVRegulationApproval', 'EVChargeCableCross']] == '').any()):
        df.loc[i, 'Section10'] = 'Invalid'

    # if answer is "Yes" and no explanation given, invalid
    elif (df.loc[i, 'EVChargeCableCross'] == "Yes") & (df.loc[i, 'CrossDetail'] == ''):
        df.loc[i, 'Section10'] = 'Invalid'
        
    # if answer is "To follow", follow-up check
    elif (df.loc[i, 'EVRegulationApproval'] == 'To follow'):
        df.loc[i, 'Section10'] = 'Follow-up Check'

    else:
        df.loc[i, 'Section10'] = 'Valid'

In [47]:
df['Section10'].value_counts()

Section10
Valid              448
Invalid            434
Follow-up Check    118
Name: count, dtype: int64

# Section 11 Rules

## Section 11 Eletrical

In [48]:
for i, x in enumerate(df['Section11Elec']):
    # if any of these columns are empty, invalid
    if (df.loc[i, ['ElecInstallation', 'InstallationCert', 'EICR']] == '').any():
        df.loc[i, 'Section11Elec'] = 'Invalid'

    # if answer is "Yes" and following column is empty, invalid
    elif (df.loc[i, 'ElecInstallation'] == 'Yes') & (df.loc[i, 'InstallationDetail'] == ''):
        df.loc[i, 'Section11Elec'] = 'Invalid'

    elif (df.loc[i, 'InstallationCert'] == 'Yes') & (df.loc[i, 'CertCopy'] == ''):
        df.loc[i, 'Section11Elec'] = 'Invalid'

    elif (df.loc[i, 'EICR'] == 'Yes') & (df.loc[i, 'EICRCopy'] == ''):
        df.loc[i, 'Section11Elec'] = 'Invalid'

    # if any of these columns' answer is "To follow", follow-up check
    elif (df.loc[i, ['CertCopy', 'EICRCopy']] == 'To follow').any():
        df.loc[i, 'Section11Elec'] = 'Follow-up Check'

    else:
        df.loc[i, 'Section11Elec'] = 'Valid'

In [49]:
df["Section11Elec"].value_counts()

Section11Elec
Invalid            424
Valid              304
Follow-up Check    272
Name: count, dtype: int64

## Section 11 Heat

In [50]:
# these are the choices on how the property was heated
sect11heat = ['Mains gas', 'Oil', ' Heat pumps', 'Liquid gas', 'Electricity', 'Underfloor', 'Woodburning/multi-fuel stove', 'Other']

# these are the detail columns for the heating system that should not be empty
impsect11 = ['HeatingInstall', 'BoilerInstall', 'HeatingReplace', 'HeatingAltCert', 'InspectionCopy', 'HeatingBoilerWork', 'HeatingBoilerYear', 'MultipleHeatingSystem']

In [51]:
for i, x in enumerate(df['Section11Heat']):
    # if none of the heat system choices are selected, invalid
    if not any(heat_choice in str(df.loc[i, 'PropertyHeating']) for heat_choice in sect11heat):
        df.loc[i, 'Section11Heat'] = 'Invalid'

    # if answer is "Other" and no explanation given, invalid
    elif ('Other' in str(df.loc[i, 'PropertyHeating'])) &  (df.loc[i, 'OtherHeatingDetail'] == ''):
        df.loc[i, 'Section11Heat'] = 'Invalid'

    # if any of these columns are empty, invalid
    elif (df.loc[i, impsect11] == '').any():
        df.loc[i, 'Section11Heat'] = 'Invalid'

    # if any of these columns' answer is "Not known", follow-up check
    elif (df.loc[i, ['HeatingInstall', 'BoilerInstall', 'HeatingReplace', 'HeatingBoilerWork', 'HeatingBoilerYear']] == 'Not known').any():
        df.loc[i, 'Section11Heat'] = 'Follow-up Check'

    # if any of these columns' answer is "To follow", follow-up check
    elif (df.loc[i, ['HeatingAltCert', 'InspectionCopy', 'MultipleHeatingSystem']] == 'To follow').any():
        df.loc[i, 'Section11Heat'] = 'Follow-up Check'

    # if column answer is non-empty and following column is "To follow", follow-up check
    elif (df.loc[i, 'HeatingBoilerYear'] != '') & (df.loc[i, 'ServiceReportCopy'] == 'To follow'):
        df.loc[i, 'Section11Heat'] = 'Follow-up Check'

    # if answer is "To follow", follow-up check
    elif (df.loc[i, 'MultipleHeatingSystem'] == 'To follow'):
        df.loc[i, 'Section11Heat'] = 'Follow-up Check'

    else:
        df.loc[i, 'Section11Heat'] = 'Valid'

In [52]:
df['Section11Heat'].value_counts()

Section11Heat
Invalid            780
Follow-up Check    218
Valid                2
Name: count, dtype: int64

In [53]:
# Double checking to see if "Valid" exists in the output column
check6 = [i for i, x in enumerate(df['Section11Heat']) if x == 'Valid']
print(check6)

[403, 497]


## Section 11 Sewer

In [54]:
# these columns should not be empty
impsect11_2 = ['FoulWaterDrain', 'SurfaceWaterDrain', 'SepticTank', 'SewageTreatment', 'Cesspool']

# these explanation columns shold not be empty under a certain condition
sect11sewer = ['DischargeDate', 'SewerageInstallDate', 'ReplacementDate', 'CesspoolEmptiedDate', 'TreatmentDate', 'WhichDischarge', 'SewerageShared', 'SewerageOutsideBound', 'MiscDocument']

In [55]:
for i, x in enumerate(df['Section11Sewer']):
    # if any of these columns are empty, invalid
    if (df.loc[i, impsect11_2] == '').any():
        df.loc[i, 'Section11Sewer'] = 'Invalid'

    # if any of the columns' answer are "Yes" and any of the following columns are empty, invalid
    elif ((df.loc[i, ['SepticTank', 'SewageTreatment', 'Cesspool']] == 'Yes').any()) &  ((df.loc[i, sect11sewer] == '').any()):
        df.loc[i, 'Section11Sewer'] = 'Invalid'

    # if answer is "Ground" and the following column is empty, invalid
    elif (df.loc[i, 'WhichDischarge'] == 'Ground') & (df.loc[i, 'InfiltrationSystem'] == ''):
        df.loc[i, 'Section11Sewer'] = 'Invalid'

    # if answer is "Yes" and no explanation given, invalid
    elif (df.loc[i, 'SewerageShared'] == 'Yes') & (df.loc[i, 'SharedDetail'] == ''):
        df.loc[i, 'Section11Sewer'] = 'Invalid'

    # if answer is "Yes" and the following column is empty, invalid
    elif (df.loc[i, 'SewerageOutsideBound'] == 'Yes') & (df.loc[i, 'OutsideBoundPlan'] == ''):
        df.loc[i, 'Section11Sewer'] = 'Invalid'
    
    # if any of the columns' answer is "Not known", follow-up check
    elif (df.loc[i, ['FoulWaterDrain', 'SurfaceWaterDrain', 'DischargeDate']] == 'Not known').any():
        df.loc[i, 'Section11Sewer'] = 'Follow-up Check'

    # if answer is "Yes" and the following column answer's is "To follow", follow-up check
    elif (df.loc[i, 'SewerageOutsideBound'] == 'Yes') & (df.loc[i, 'OutsideBoundPlan'] == 'To follow'):
        df.loc[i, 'Section11Sewer'] = 'Follow-up Check'

    # if ansiwer is "To follow", follow-up check
    elif (df.loc[i, 'MiscDocument'] == 'To follow'):
        df.loc[i, 'Section11Sewer'] = 'Follow-up Check'

    else:
        df.loc[i, 'Section11Sewer'] = 'Valid'

In [56]:
df['Section11Sewer'].value_counts()

Section11Sewer
Invalid            486
Follow-up Check    419
Valid               95
Name: count, dtype: int64

In [57]:
# Double checking to see if "Valid" exists in the output column
checking = [i for i, x in enumerate(df['Section11Sewer']) if x == 'Valid']
print(checking)

[0, 1, 2, 3, 11, 14, 15, 28, 37, 38, 43, 45, 48, 51, 61, 63, 72, 78, 80, 87, 91, 99, 108, 111, 113, 115, 125, 154, 155, 156, 157, 158, 165, 166, 173, 180, 181, 183, 187, 192, 196, 199, 203, 214, 216, 219, 225, 228, 239, 245, 248, 261, 279, 282, 290, 300, 306, 321, 323, 332, 336, 337, 340, 343, 349, 356, 359, 361, 368, 388, 390, 393, 402, 405, 407, 419, 434, 436, 437, 438, 446, 450, 454, 456, 461, 463, 468, 472, 479, 486, 496, 576, 586, 731, 749]


# Section 12 Rules

In [58]:
for i, x in enumerate(df['Section12Elec']):
    # if answer is empty, invalid
    if df.loc[i, 'MainElec'] == '':
        df.loc[i, 'Section12Elec'] = 'Invalid'
        
    # if answer is 'Yes' and no details given, invalid
    elif (df.loc[i, 'MainElec'] == 'Yes') & ((df.loc[i, ['ElecProvider', 'ElecMeter', 'MPAN']] == '').any()):
        df.loc[i, 'Section12Elec'] = 'Invalid'

    else:
        df.loc[i, 'Section12Elec'] = 'Valid'

In [59]:
df['Section12Elec'].value_counts()

Section12Elec
Valid      686
Invalid    314
Name: count, dtype: int64

In [60]:
for i, x in enumerate(df['Section12Gas']):
    # if answer is empty, invalid
    if df.loc[i, 'MainGas'] == '':
        df.loc[i, 'Section12Gas'] = 'Invalid'
        
    # if answer is 'Yes' and no details given, invalid
    elif (df.loc[i, 'MainGas'] == 'Yes') & ((df.loc[i, ['GasProvider', 'GasMeter', 'MPRN']] == '').any()):
        df.loc[i, 'Section12Gas'] = 'Invalid'

    else:
        df.loc[i, 'Section12Gas'] = 'Valid'

In [61]:
df['Section12Gas'].value_counts()

Section12Gas
Valid      700
Invalid    300
Name: count, dtype: int64

In [62]:
for i, x in enumerate(df['Section12Water']):
    # if answer is empty, invalid
    if df.loc[i, 'MainWater'] == '':
        df.loc[i, 'Section12Water'] = 'Invalid'
        
    # if answer is 'Yes' and no details given, invalid
    elif (df.loc[i, 'MainWater'] == 'Yes') & ((df.loc[i, ['WaterProvider', 'WaterStopcock', 'WaterMeter']] == '').any()):
        df.loc[i, 'Section12Water'] = 'Invalid'

    else:
        df.loc[i, 'Section12Water'] = 'Valid'

In [63]:
df['Section12Water'].value_counts()

Section12Water
Valid      682
Invalid    318
Name: count, dtype: int64

In [64]:
for i, x in enumerate(df['Section12Sewage']):
    # if any of these columns are empty, invalid
    if (df.loc[i, ['MainSewerage', 'SewageTreatmentPlant']] == '').any():
        df['Section12Sewage'] = 'Invalid'

    # if answer is 'Yes' and no details given, invalid
    elif (df.loc[i, 'MainSewerage'] == 'Yes') & (df.loc[i, 'SewerageProvider'] == ''):
        df['Section12Sewage'] = 'Invalid'

    elif (df.loc[i, 'SewageTreatmentPlant'] == 'Yes') & ((df.loc[i, ['SewageTreatmentProvider', 'SewageTreatmentModel', 'SewageServiceProvider']] == '').any()):
        df.loc[i, 'Section12Sewage'] = 'Invalid'

    else:
        df.loc[i, 'Section12Sewage'] = 'Valid'

In [65]:
df['Section12Sewage'].value_counts()

Section12Sewage
Invalid    1000
Name: count, dtype: int64

In [66]:
for i, x in enumerate(df['Section12Heat']):
    # if answer is empty, invalid
    if df.loc[i, 'HeatPumps'] == '':
        df.loc[i, 'Section12Heat'] = 'Invalid'
        
    # if answer is 'Yes' and no details given, invalid
    elif (df.loc[i, 'HeatPumps'] == 'Yes') & ((df.loc[i, ['HeatPumpsProvider', 'HeatPumpsModel', 'HeatPumpsServiceProvider']] == '').any()):
        df.loc[i, 'Section12Heat'] = 'Invalid'

    else:
        df.loc[i, 'Section12Heat'] = 'Valid'

In [67]:
df['Section12Heat'].value_counts()

Section12Heat
Valid      697
Invalid    303
Name: count, dtype: int64

In [68]:
for i, x in enumerate(df['Section12Tele']):
    # if any of these columns are empty, invalid
    if (df.loc[i, ['Telephone', 'Broadband']] == '').any():
        df.loc[i, 'Section12Tele'] = 'Invalid'

    # if answer is "Yes" and no detail given, invalid
    elif (df.loc[i, 'Telephone'] == 'Yes') & (df.loc[i, 'TelephoneProvider'] == ''):
        df.loc[i, 'Section12Tele'] = 'Invalid'

    elif (df.loc[i, 'Broadband'] == 'Yes') & (df.loc[i, 'BroadbandProvider'] == ''):
        df.loc[i, 'Section12Tele'] = 'Invalid'

    else:
        df.loc[i, 'Section12Tele'] = 'Valid'

In [69]:
df['Section12Tele'].value_counts()

Section12Tele
Valid      623
Invalid    377
Name: count, dtype: int64

# Section 13

In [70]:
# none of these columns should be empty
impsect13 = ['AnotherPropertySale', 'SpecialRequirement', 'LiveAtProperty', 'Age17OrOver', 'VacantPossession', 'SaleContractSign', 'OccupiersDetailAndCopy']

In [71]:
for i, x in enumerate(df['Section13']):
    # if any of these columns are empty, invalid
    if (df.loc[i, impsect13] == '').any():
        df.loc[i, 'Section13'] = 'Invalid'

    # if answer is "Yes" and no explanation given, invalid
    elif (df.loc[i, 'SpecialRequirement'] == 'Yes') & (df.loc[i, 'RequirementDetail'] == ''):
        df.loc[i, 'Section13'] = 'Invalid'

    # if answer is "Yes" and any of the following column is empty, invalid
    elif (df.loc[i, 'Age17OrOver'] == 'Yes') & ((df.loc[i, ['Age17OverName', 'Age17OverTenant']] == '').any()):
        df.loc[i, 'Section13'] = 'Invalid'

    # if answer is "To follow", follow-up check
    elif df.loc[i, 'OccupiersDetailAndCopy'] == 'To follow':
        df.loc[i, 'Section13'] = 'Follow-up Check'

    else:
        df.loc[i, 'Section13'] = 'Valid'

In [72]:
df['Section13'].value_counts()

Section13
Invalid            491
Follow-up Check    258
Valid              251
Name: count, dtype: int64

# Section 14

In [73]:
# these are the columns that should not be empty
impsect14 = ['SalePriceMortage', 'RubbishRemoved', 'FittingsReplaced', 'OtherFittings', 'KeysAndCodes']

In [74]:
for i, x in enumerate(df['Section14']):
    # if any of these columns are empty, invalid
    if (df.loc[i, impsect14] == '').any():
        df.loc[i, 'Section14'] = 'Invalid'

    # if the answer is 'No', invalid
    elif df.loc[i, 'SalePriceMortage'] == 'No':
        df.loc[i, 'Section14'] = 'Follow-up Check'

    # if any of these columns' answer is "No", follow-up check
    elif (df.loc[i, ['RubbishRemoved', 'FittingsReplaced', 'OtherFittings', 'KeysAndCodes']] == 'No').any():
        df.loc[i, 'Section14'] = 'Follow-up Check'

    else:
        df.loc[i, 'Section14'] = 'Valid'

In [75]:
df['Section14'].value_counts()

Section14
Follow-up Check    551
Invalid            425
Valid               24
Name: count, dtype: int64

# Section 15

In [76]:
# these are the choices allowed within the question
impsect15 = ['Attached', 'To follow', 'Not applicable', 'Not available']

In [77]:
for i, x in enumerate(df['Section15']):
    # if none of these choices are included, invalid
    if not any(copy_choice in str(df.loc[i, 'ConsentsCopies']) for copy_choice in impsect15):
        df.loc[i, 'Section15'] = 'Invalid'

    # if "Attached" was selected and no indication was given, invalid
    elif ('Attached' in df.loc[i, 'ConsentsCopies']) & (not('Consents attached' in df.loc[i, 'ConsentsAttached&ToFollow'])):
        df.loc[i, 'Section15'] = 'Invalid'

    # if "To follow" was selected and no indication was given, invalid
    elif ('To follow' in df.loc[i, 'ConsentsCopies']) & (not('Consents to follow' in df.loc[i, 'ConsentsAttached&ToFollow'])):
        df.loc[i, 'Section15'] = 'Invalid'

    # if "Not available" was selected and no indication was given, invalid
    elif ('Not available' in df.loc[i, 'ConsentsCopies']) & (df.loc[i, 'ConsentsNotAvailable'] == ''):
        df.loc[i, 'Section15'] = 'Invalid'

    # if "To follow" is the answer, follow-up check
    elif "To follow" in str(df.loc[i, 'ConsentsCopies']):
        df.loc[i, 'Section15'] = 'Follow-up Check'

    elif 'To follow' in str(df.loc[i, 'FurtherInformationDocuments']):
        df.loc[i, 'Section15'] = 'Follow-up Check'

    else:
        df.loc[i, 'Section15'] = 'Valid'

In [78]:
df['Section15'].value_counts()

Section15
Follow-up Check    448
Invalid            341
Valid              211
Name: count, dtype: int64

In [79]:
invalid_check = [i for i, x in enumerate(df['Section15']) if x == 'Invalid']
print(invalid_check)

[36, 37, 42, 55, 73, 98, 101, 119, 153, 175, 180, 187, 199, 224, 226, 281, 290, 300, 331, 334, 337, 364, 372, 383, 398, 402, 410, 412, 416, 434, 500, 501, 503, 507, 508, 511, 512, 514, 515, 517, 519, 520, 521, 523, 524, 525, 528, 531, 532, 533, 534, 538, 541, 543, 546, 548, 551, 552, 553, 555, 556, 559, 560, 561, 565, 566, 567, 568, 569, 571, 572, 573, 575, 576, 578, 579, 580, 581, 583, 586, 589, 591, 593, 594, 595, 596, 598, 599, 601, 602, 603, 605, 606, 607, 608, 609, 612, 613, 614, 615, 616, 617, 618, 619, 622, 623, 624, 625, 626, 627, 629, 630, 632, 633, 635, 637, 639, 641, 642, 643, 644, 645, 646, 647, 651, 652, 653, 654, 655, 657, 658, 659, 660, 661, 664, 665, 666, 669, 670, 671, 672, 673, 674, 675, 676, 677, 678, 679, 680, 681, 684, 686, 691, 692, 693, 695, 696, 698, 699, 700, 701, 702, 703, 706, 708, 709, 710, 711, 713, 714, 717, 718, 719, 721, 725, 727, 729, 733, 735, 736, 738, 739, 742, 745, 746, 747, 748, 749, 750, 751, 754, 755, 756, 757, 758, 759, 760, 761, 762, 763, 769, 

In [81]:
df.to_csv("DissertationCompletedDataset.csv", index = False)